Step 0, download the EIA 860 and EIA 923 zip files for the year you are intrested in from the the EIA website. then, change the year var to the year you are intrested in.

In [ ]:
import pandas as pd
import numpy as np
#from IPython.display import HTML
year = 2024

Step 1, load in the EIA files straight from their zip file. 

In [ ]:
plant_860 = pd.read_excel("/Users/mshaaban/clab_underground_th_storage/Finding_IRL_MISO_CFs/eia860"+str(year)+"/2___Plant_Y"+str(year)+".xlsx", skiprows=1)
generator_860 = pd.read_excel("/Users/mshaaban/clab_underground_th_storage/Finding_IRL_MISO_CFs/eia860"+str(year)+"/3_1_Generator_Y"+str(year)+".xlsx", sheet_name="Operable" , skiprows=1)
#eia923 = pd.read_excel("/Users/mshaaban/clab_underground_th_storage/Finding_IRL_MISO_CFs/f923_"+str(year)+"/EIA923_Schedules_2_3_4_5_M_12_"+str(year)+"_Final_Revision.xlsx", sheet_name="Page 1 Generation and Fuel Data", skiprows=5)

In [ ]:
eia923 = pd.read_excel("/Users/mshaaban/clab_underground_th_storage/Finding_IRL_MISO_CFs/f923_2024/EIA923_Schedules_2_3_4_5_M_12_2024_Final.xlsx", sheet_name="Page 1 Generation and Fuel Data", skiprows=5)

Step 2, filter the files,

In [ ]:
plant_860[plant_860["Balancing Authority Code"] == "MISO"].to_excel("plants_only_miso_"+str(year)+".xlsx", index=True)
generator_860[((generator_860["Energy Source 1"] == "WND") | (generator_860["Energy Source 1"] == "SUN")) & (generator_860["Status"] == "OP")].to_excel("generators_only_pv_wind_"+str(year)+".xlsx", index=True)
eia923[(((eia923["Reported\nFuel Type Code"] == "WND") | (eia923["Reported\nFuel Type Code"] == "SUN")) & (eia923["Balancing\nAuthority Code"] == "MISO"))].to_excel("eia923_refined"+str(year)+".xlsx", index=True)


In [ ]:
miso_plants = pd.read_excel("plants_only_miso_"+str(year)+".xlsx")
pv_wind_gens = pd.read_excel("generators_only_pv_wind_"+str(year)+".xlsx")
eia923_refined = pd.read_excel("eia923_refined"+ str(year)+".xlsx")

In [ ]:
netgen_cols = [
    "Netgen\nJanuary", "Netgen\nFebruary", "Netgen\nMarch", "Netgen\nApril",
    "Netgen\nMay",     "Netgen\nJune",     "Netgen\nJuly",  "Netgen\nAugust",
    "Netgen\nSeptember","Netgen\nOctober", "Netgen\nNovember","Netgen\nDecember"
]

for col in netgen_cols:
    eia923_refined[col] = pd.to_numeric(eia923_refined[col], errors="coerce").fillna(0)
    
eia923_refined.to_excel("eia923_refined" + str(year) + ".xlsx", index=False)


In [ ]:
results = []
check = []
merged_row = 0

for i in range(len(miso_plants)):
    plant_code = miso_plants["Plant Code"].iloc[i]
    matches = pv_wind_gens[pv_wind_gens["Plant Code"] == plant_code]

    if not matches.empty:
     # Single generator — simple combine

        if(len(matches) == 1):
            plant_row = miso_plants.iloc[[i]].reset_index(drop=True)
            for _, gen_row in matches.iterrows():
                gen_row_df = pd.DataFrame([gen_row]).reset_index(drop=True)
                combined = pd.concat([gen_row_df, plant_row], axis=1)
                results.append(combined)
                merged_row += 1
        else:
            wind = []
            solar = []
            for j in range(len(matches)):
                prime_mover = matches["Prime Mover"].iloc[j]
                fuel = matches["Energy Source 1"].iloc[j]
                if ((matches["Prime Mover"].iloc[j] == 'WT' or fuel == "WND")):
                        wind.append(matches.iloc[j])
                    
                elif ((matches["Prime Mover"].iloc[j] == 'PV' or fuel == "SUN")):
                        solar.append(matches.iloc[j])
                
            if wind:
                wind_df = pd.DataFrame(wind).reset_index(drop=True)
                wind_agg = wind_df.iloc[[0]].copy()
                wind_agg["Summer Capacity (MW)"] = pd.to_numeric(wind_df["Summer Capacity (MW)"], errors="coerce").fillna(0).sum()
                plant_row = miso_plants.iloc[[i]].reset_index(drop=True)
                combined  = pd.concat([wind_agg.reset_index(drop=True), plant_row], axis=1)
                results.append(combined)
                merged_row += 1

            if solar:                                                                        # ✅ guard
                solar_df = pd.DataFrame(solar).reset_index(drop=True)
                solar_agg = solar_df.iloc[[0]].copy()
                solar_agg["Summer Capacity (MW)"] = pd.to_numeric(solar_df["Summer Capacity (MW)"], errors="coerce").fillna(0).sum()  # ✅ was wind_df
                plant_row = miso_plants.iloc[[i]].reset_index(drop=True)
                combined  = pd.concat([solar_agg.reset_index(drop=True), plant_row], axis=1)
                results.append(combined)
                merged_row += 1



final = pd.concat(results, ignore_index=True)
final.to_excel('merged_MISO_win_sun_'+str(year)+'.xlsx', index=False)
print("Total matches:", merged_row)

In [ ]:
merged_miso_renewable = pd.read_excel("merged_MISO_win_sun_" + str(year) + ".xlsx")
eia923_refined = pd.read_excel("eia923_refined"+str(year)+".xlsx")

merged_miso_renewable[merged_miso_renewable['Prime Mover'] == "PV"].to_excel("merged_miso_solar"+str(year)+".xlsx", index=True)
merged_miso_renewable[merged_miso_renewable['Prime Mover'] == "WT"].to_excel("merged_miso_wind"+str(year)+".xlsx", index=True)

eia923_refined[eia923_refined['Reported\nPrime Mover'] == "PV"].to_excel("eia923_miso_solar"+str(year)+".xlsx", index=True)
eia923_refined[eia923_refined['Reported\nPrime Mover'] == "WT"].to_excel("eia923_miso_wind"+str(year)+".xlsx", index=True)

eia923_refined_wind = pd.read_excel("eia923_miso_wind"+str(year)+".xlsx")
eia923_refined_solar = pd.read_excel("eia923_miso_solar"+str(year)+".xlsx")

merged_miso_wind = pd.read_excel("Merged_miso_wind"+str(year)+".xlsx")
merged_miso_solar = pd.read_excel("Merged_miso_solar"+str(year)+".xlsx")

In [ ]:
merged_miso_renewable = pd.read_excel("merged_MISO_win_sun_" + str(year) + ".xlsx")

results    = []
merged_row = 0

for i in range(len(merged_miso_renewable)):
    plant_code = merged_miso_renewable["Plant Code"].iloc[i]
    fuel_type  = merged_miso_renewable["Energy Source 1"].iloc[i]

    # ✅ Match on both Plant Id AND fuel type to avoid cross-fuel contamination
    matches = eia923_refined[
        (eia923_refined["Plant Id"] == plant_code) &
        (eia923_refined["Reported\nFuel Type Code"]  == fuel_type)
    ]

    if matches.empty:
        continue

    # Coerce netgen columns to numeric
    matches = matches.copy()
    for col in netgen_cols:
        if col in matches.columns:
            matches[col] = pd.to_numeric(matches[col], errors="coerce").fillna(0)

    if len(matches) == 1:
        # Single EIA-923 row — simple combine
        gen_row_df = pd.DataFrame([matches.iloc[0]]).reset_index(drop=True)
        plant_row  = merged_miso_renewable.iloc[[i]].reset_index(drop=True)
        combined   = pd.concat([gen_row_df, plant_row], axis=1)
        results.append(combined)
        merged_row += 1

    else:
        # Multiple EIA-923 rows for same plant+fuel — aggregate into one
        agg_row = pd.DataFrame([matches.iloc[0]]).reset_index(drop=True).copy()
        for col in netgen_cols + ["Summer Capacity (MW)"]:
            if col in matches.columns:
                agg_row[col] = pd.to_numeric(matches[col], errors="coerce").fillna(0).sum()

        plant_row = merged_miso_renewable.iloc[[i]].reset_index(drop=True)
        combined  = pd.concat([agg_row.reset_index(drop=True), plant_row], axis=1)
        results.append(combined)
        merged_row += 1

final = pd.concat(results, ignore_index=True)
final = final.loc[:, ~final.columns.duplicated()]  # drop any duplicate columns from concat
final.to_excel("merged_miso_renew_eia923_" + str(year) + ".xlsx", index=False)
print("Total matches:", merged_row)

GOOD Up Until here

In [ ]:
NetGen_year = 0
NameCap = 1

netgen_cols = [
    "Netgen\nJanuary", "Netgen\nFebruary", "Netgen\nMarch", "Netgen\nApril",
    "Netgen\nMay",     "Netgen\nJune",     "Netgen\nJuly",  "Netgen\nAugust",
    "Netgen\nSeptember","Netgen\nOctober", "Netgen\nNovember","Netgen\nDecember"
]

netgen_year_wind = dict.fromkeys(eia923_refined_wind['Plant Id'].unique(), 0)
netgen_year_solar = dict.fromkeys(eia923_refined_solar['Plant Id'].unique(), 0)

nameplate_cap_wind = dict.fromkeys(merged_miso_wind['Plant Code'].unique(), 1)
nameplate_cap_solar = dict.fromkeys(merged_miso_solar['Plant Code'].unique(), 1)

for i in range(len(eia923_refined_wind)):
    working_row = eia923_refined_wind.iloc[i]
    NetGen_year = working_row[netgen_cols].sum()
    PlantID = working_row['Plant Id']
    netgen_year_wind[PlantID] = NetGen_year

for i in range(len(eia923_refined_solar)):
    working_row = eia923_refined_solar.iloc[i]
    NetGen_year = working_row[netgen_cols].sum()
    PlantID = working_row['Plant Id']
    netgen_year_solar[PlantID] = NetGen_year

for i in range(len(merged_miso_wind)):
    working_row = merged_miso_wind.iloc[i]
    NameCap = working_row['Nameplate Capacity (MW)']
    PlantCode = working_row['Plant Code']
    nameplate_cap_wind[PlantCode] = NameCap

for i in range(len(merged_miso_solar)):
    working_row = merged_miso_solar.iloc[i]
    NameCap = working_row['Nameplate Capacity (MW)']
    PlantCode = working_row['Plant Code']
    nameplate_cap_solar[PlantCode] = NameCap




In [ ]:
for key in (list(netgen_year_wind.keys())):
    if netgen_year_wind[key] == 0:
        del netgen_year_wind[key]
        if key in nameplate_cap_wind:
            del nameplate_cap_wind[key]
            
for key in (list(netgen_year_solar.keys())):
    if netgen_year_solar[key] == 0:
        del netgen_year_solar[key]
        if key in nameplate_cap_solar:
            del nameplate_cap_solar[key]

In [ ]:
print('solar')
print(len(list(nameplate_cap_solar.keys())))
print(len(list(netgen_year_solar.keys())))

if len(list(netgen_year_solar.keys())) <= len(list(nameplate_cap_solar.keys())):
    key_box_solar = list(netgen_year_solar.keys())
else:
    key_box_solar = list(nameplate_cap_solar.keys())

print('wind')
print(len(list(netgen_year_wind.keys())))
print(len(list(nameplate_cap_wind.keys())))

if len(list(netgen_year_wind.keys())) <= len(list(nameplate_cap_wind.keys())):
    key_box_wind = list(netgen_year_wind.keys())
else:
    key_box_wind = list(nameplate_cap_wind.keys())

In [ ]:
cap_factor_wind = dict.fromkeys(key_box_wind, 0)

for key in cap_factor_wind:
    if key not in netgen_year_wind or key not in nameplate_cap_wind:
        continue
    netgenX = netgen_year_wind[key]
    capacityX = nameplate_cap_wind[key]

    CapacityFactor = netgenX / (capacityX * 8760)
    if CapacityFactor <= 1:
        cap_factor_wind[key] = CapacityFactor
 
avg_list_capfac_wind = []
for key in cap_factor_wind:
    if cap_factor_wind[key] != 0:
        avg_list_capfac_wind.append(cap_factor_wind[key])

cap_factor_solar = dict.fromkeys(key_box_solar, 0)

for key in cap_factor_solar:
    if key not in netgen_year_solar or key not in nameplate_cap_solar:
        continue
   
    netgenX = netgen_year_solar[key]
    capacityX = nameplate_cap_solar[key]

    CapacityFactor = ((netgenX)/(capacityX*8760))
    
    if (CapacityFactor <= 1):
        cap_factor_solar[key] = CapacityFactor
        
avg_list_capfac_solar = []  
for key in cap_factor_solar:
    if cap_factor_solar[key] != 0:
        avg_list_capfac_solar.append(cap_factor_solar[key])

import statistics
print('wind CF')
print(statistics.mean(avg_list_capfac_wind))
print('solar CF')
print(statistics.mean(avg_list_capfac_solar))

In [ ]:
print((0.3166+0.3213+0.3665+0.3175+0.3315)/5)

In [ ]:
print((0.2038+0.2226+0.2303+0.2283+0.2181)/5)